# 04 - Continue SFT-BE on cleaned full AllNLI with early stopping

Notebook nay chay doc lap tren Kaggle. Upload checkpoint `stage0_final.pt` vao Kaggle Dataset, bam Add Input cho notebook, code se tu tim trong `/kaggle/input`.

In [1]:
from pathlib import Path

PROJECT_ROOT = Path('/kaggle/working/similarity_search')
GITHUB_REPOSITORY_URL = 'https://github.com/PhDQuang/similarity_search.git'

if not PROJECT_ROOT.exists():
    !git clone {GITHUB_REPOSITORY_URL} {PROJECT_ROOT}

%cd {PROJECT_ROOT}
%pip install -q -r fix/requirements-kaggle.txt
%pip install -q -e fix
%pip install -q -e .

Cloning into '/kaggle/working/similarity_search'...
remote: Enumerating objects: 222, done.
remote: Counting objects: 100% (222/222), done.
remote: Compressing objects: 100% (155/155), done.
remote: Total 222 (delta 90), reused 185 (delta 53), pack-reused 0 (from 0)
Receiving objects: 100% (222/222), 481.65 KiB | 3.98 MiB/s, done.
Resolving deltas: 100% (90/90), done.
/kaggle/working/similarity_search
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 639.7 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 54.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 37.9 MB/s eta 0:00:00
Note: you may need to restart the kernel to use updated packages.
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for similarity-search-fix (pyproject.toml) ... done
Note: yo

In [2]:
from pathlib import Path
import shutil

CLEAN_DATA_DIR = Path('fix/data/processed/allnli_70_15_15_clean/pair-class')
KAGGLE_CLEAN_CANDIDATES = [
    Path('/kaggle/input/allnli-70-15-15-clean/pair-class'),
    Path('/kaggle/input/allnli-70-15-15-clean/allnli_70_15_15_clean/pair-class'),
]

def has_clean_data(path: Path) -> bool:
    return all((path / f'{split}.parquet').exists() for split in ('train', 'val', 'test'))

if not has_clean_data(CLEAN_DATA_DIR):
    source = next((path for path in KAGGLE_CLEAN_CANDIDATES if has_clean_data(path)), None)
    if source is not None:
        CLEAN_DATA_DIR.mkdir(parents=True, exist_ok=True)
        for item in source.iterdir():
            if item.is_file():
                shutil.copy2(item, CLEAN_DATA_DIR / item.name)
    else:
        !python -m similarity_search_fix.data.prepare_allnli_70_15_15_clean --output-dir {CLEAN_DATA_DIR} --seed 42

assert has_clean_data(CLEAN_DATA_DIR), f'Missing clean data: {CLEAN_DATA_DIR}'
print('Using clean data:', CLEAN_DATA_DIR)

README.md: 5.15kB [00:00, 22.2MB/s]
pair-class/train-00000-of-00001.parquet: 100%|█| 69.5M/69.5M [00:16<00:00, 4.18M
pair-class/dev-00000-of-00001.parquet: 100%|█| 1.57M/1.57M [00:02<00:00, 780kB/s
pair-class/test-00000-of-00001.parquet: 100%|█| 1.61M/1.61M [00:00<00:00, 2.63MB
Generating train split: 100%|█| 942069/942069 [00:00<00:00, 1464855.17 examples/
Generating test split: 100%|██| 19656/19656 [00:00<00:00, 1339407.97 examples/s]
{
  "dataset_name": "sentence-transformers/all-nli",
  "dataset_config": "pair-class",
  "created_at_utc": "2026-07-05T11:25:34.267121+00:00",
  "seed": 42,
  "split_ratios": {
    "train": 0.7,
    "val": 0.15,
    "test": 0.15
  },
  "saved_paths": {
    "train": "fix/data/processed/allnli_70_15_15_clean/pair-class/train.parquet",
    "val": "fix/data/processed/allnli_70_15_15_clean/pair-class/val.parquet",
    "test": "fix/data/processed/allnli_70_15_15_clean/pair-class/test.parquet"
  },
  "row_counts": {
    "train": 686442,
    "val": 147033,
    

In [3]:
from pathlib import Path

SFTBE_CHECKPOINT_CANDIDATES = [
    Path('/kaggle/input/sftbe-stage0/stage0_final.pt'),
    Path('/kaggle/input/sftbe-checkpoint/stage0_final.pt'),
    Path('/kaggle/input/sftbe_checkpoint/stage0_final.pt'),
    Path('models/sftbe_checkpoint/stage0_final.pt'),
]
SFTBE_CHECKPOINT_CANDIDATES.extend(Path('/kaggle/input').glob('**/stage0_final.pt'))
SFTBE_CHECKPOINT_CANDIDATES.extend(Path('/kaggle/input').glob('**/stage0*.pt'))
SFTBE_CHECKPOINT_CANDIDATES.extend(Path('/kaggle/input').glob('**/sftbe*.pt'))

SFTBE_CHECKPOINT_PATH = next(
    (path for path in SFTBE_CHECKPOINT_CANDIDATES if path.exists()),
    None,
)
if SFTBE_CHECKPOINT_PATH is None:
    found_pt_files = sorted(str(path) for path in Path('/kaggle/input').glob('**/*.pt'))[:30]
    raise FileNotFoundError(
        'Khong tim thay SFT-BE Stage 0 checkpoint. Upload stage0_final.pt vao Kaggle Dataset, '
        'attach dataset do vao notebook, hoac sua SFTBE_CHECKPOINT_CANDIDATES. '
        f'Found .pt files: {found_pt_files}'
    )
print('Using SFT-BE checkpoint:', SFTBE_CHECKPOINT_PATH)

OUTPUT_DIR = Path('/kaggle/working/fix_outputs/sftbe_clean_full')
MODEL_DIR = Path('/kaggle/working/fix_models/sftbe_clean_full')

!python -m similarity_search_fix.models.train_sftbe \
  --input-dir {CLEAN_DATA_DIR} \
  --output-dir {OUTPUT_DIR} \
  --model-dir {MODEL_DIR} \
  --checkpoint-path {SFTBE_CHECKPOINT_PATH} \
  --num-train-epochs 5 \
  --batch-size 64 \
  --eval-batch-size 128 \
  --learning-rate 2e-5 \
  --log-every-steps 500 \
  --early-stopping-patience 2 \
  --early-stopping-min-delta 1e-4 \
  --eval-during-training-samples 20000 \
  --max-retrieval-queries 1000 \
  --test-sample-size 5000 \
  --seed 42

Using SFT-BE checkpoint: /kaggle/input/datasets/hcsinhgoethe/sftbe-stage0/stage0_final.pt
tokenizer_config.json: 100%|██████████████████| 48.0/48.0 [00:00<00:00, 392kB/s]
config.json: 100%|█████████████████████████████| 570/570 [00:00<00:00, 4.67MB/s]
vocab.txt: 232kB [00:00, 919kB/s]
tokenizer.json: 466kB [00:00, 1.17MB/s]
/kaggle/working/similarity_search/fix/src/similarity_search_fix/models/train_sftbe.py:208: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)
/kaggle/working/similarity_search/fix/src/similarity_search_fix/models/train_sftbe.py:232: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):
{'step': 500, 'epoch': 1, 'train_loss': 0.14246299229562281, 'eval_mse': 0.13540143661283294, 'learning_rate': 1.864628006712661e-06}
/kagg

In [4]:
from pathlib import Path
import json
import shutil

ARTIFACT_DIR = Path('/kaggle/working/artifacts_sftbe_clean_full')
if ARTIFACT_DIR.exists():
    shutil.rmtree(ARTIFACT_DIR)
ARTIFACT_DIR.mkdir(parents=True)
shutil.copytree(OUTPUT_DIR, ARTIFACT_DIR / 'outputs')
shutil.copytree(MODEL_DIR, ARTIFACT_DIR / 'model')
zip_path = shutil.make_archive(str(ARTIFACT_DIR), 'zip', ARTIFACT_DIR)
print('Download artifact:', zip_path)

display(json.loads((OUTPUT_DIR / 'metrics.json').read_text()))
display(json.loads((OUTPUT_DIR / 'test5k_performance.json').read_text()))

Download artifact: /kaggle/working/artifacts_sftbe_clean_full.zip


{'task': 'entailment-as-semantic-similarity',
 'fixed_dataset': 'AllNLI pair-class full 70/15/15',
 'positive_label': 'entailment',
 'model': {'name': 'SFT-BE AllNLI fine-tuned',
  'source_checkpoint': '/kaggle/input/datasets/hcsinhgoethe/sftbe-stage0/stage0_final.pt',
  'final_checkpoint': '/kaggle/working/fix_models/sftbe_clean_full/stage1_allnli_final.pt',
  'trained_in_project': True,
  'loss': 'CosineMSELoss',
  'score_mapping': {'entailment': 1.0, 'neutral': 0.5, 'contradiction': 0.0},
  'hidden_size': 768},
 'threshold_selection': {'split': 'val',
  'threshold': 0.5974185466766357,
  'best_f1': 0.709470791849552},
 'pair_classification': {'val': {'threshold': 0.5974185466766357,
   'accuracy': 0.7785123067610672,
   'precision': 0.6298390673509472,
   'recall': 0.8121527777777777,
   'f1': 0.7094707918495522,
   'average_precision': 0.7285374071006652,
   'mean_positive_score': 0.7326198816299438,
   'mean_negative_score': 0.3775843381881714,
   'roc_auc': 0.8634103648299463},
 

{'sample_rows': 5000,
 'elapsed_seconds': 19.67217060800067,
 'pairs_per_second': 254.16615683306958,
 'threshold': 0.5974185466766357,
 'pair_classification': {'threshold': 0.5974185466766357,
  'accuracy': 0.7776,
  'precision': 0.6325187969924813,
  'recall': 0.8031026252983293,
  'f1': 0.7076761303890642,
  'average_precision': 0.7369222321385636,
  'mean_positive_score': 0.7287970781326294,
  'mean_negative_score': 0.3770381808280945,
  'roc_auc': 0.8629952949403916}}